<a href="https://colab.research.google.com/github/Mogi21/nlp-engineering-portfolio/blob/main/01_classical_nlp_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import urllib.request
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk import pos_tag, ne_chunk, trigrams

# ==========================================
# 1. SETUP & DATA INGESTION
# ==========================================
# Download required NLTK resource packages
nltk.download([
    'punkt_tab',
    'stopwords',
    'wordnet',
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker_tab',
    'words'
])

# Ingest BBC Politics and Business Corpus
CORPUS_URL = "http://people.dsv.su.se/hercules/BBC_politics_business.txt"
try:
    response = urllib.request.urlopen(CORPUS_URL)
    corpus_text = response.read().decode('utf-8')
    print(f"Corpus successfully loaded. Total characters: {len(corpus_text)}")
except Exception as e:
    print(f"Error downloading corpus: {e}")
    corpus_text = ""

# ==========================================
# 2. LEXICAL PROCESSING & PREPROCESSING
# ==========================================
# Tokenization
tokens = nltk.word_tokenize(corpus_text)
initial_vocab = set(tokens)
print(f"Initial State -> Tokens: {len(tokens)} | Unique Types: {len(initial_vocab)}")

# Stopword Removal (Handling Case Insensitivity)
stop_words = set(stopwords.words('english'))

def filter_stopwords(word_list, stop_words_set):
    """Filters stopwords, checking both exact match and lowercased match."""
    return [word for word in word_list if word.lower() not in stop_words_set]

filtered_tokens = filter_stopwords(tokens, stop_words)
filtered_vocab = set(filtered_tokens)
print(f"After Stopword Filter -> Tokens: {len(filtered_tokens)} | Unique Types: {len(filtered_vocab)}")

# Lemmatization vs. Stemming Analysis
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]

print(f"Vocabulary post-Lemmatization: {len(set(lemmatized_tokens))} types")
print(f"Vocabulary post-Stemming: {len(set(stemmed_tokens))} types")

# ==========================================
# 3. NAMED ENTITY RECOGNITION (NER) & POS TAGGING
# ==========================================
# Part-of-Speech Tagging
tagged_tokens = pos_tag(tokens)

# Extract Named Entities via Chunking
def extract_named_entities(tagged_data, target_labels={'PERSON', 'ORGANIZATION', 'GPE'}):
    """Parses POS-tagged data to extract specific Named Entity chunks."""
    chunked_data = ne_chunk(tagged_data)
    entities = {'PERSON': [], 'ORGANIZATION': [], 'GPE': []}

    for node in chunked_data:
        if isinstance(node, nltk.tree.Tree) and node.label() in target_labels:
            entity_name = " ".join([leaf[0] for leaf in node.leaves()])
            entities[node.label()].append(entity_name)

    return entities

ner_results = extract_named_entities(tagged_tokens)

print("\nNamed Entity Counts:")
for entity_type, entity_list in ner_results.items():
    print(f"- {entity_type}: {len(entity_list)} instances ({len(set(entity_list))} unique)")

# ==========================================
# 4. N-GRAM ANALYSIS
# ==========================================
# Extract continuous Trigrams to identify common multi-word phrases
corpus_trigrams = list(trigrams(filtered_tokens))
trigram_freq = Counter(corpus_trigrams)

print("\nTop 5 Most Frequent Trigrams:")
for gram, count in trigram_freq.most_common(5):
    print(f"- {' '.join(gram)}: {count}")

# ==========================================
# 5. CONTEXT-AWARE REGEX EXTRACTION
# ==========================================
def extract_keyword_context(text, keyword, window_size=30):
    """
    Extracts occurrences of a keyword along with surrounding contextual characters.
    Useful for quick entity relation checks without heavy dependency parsing.
    """
    pattern = rf".{{0,{window_size}}}\b{keyword}\b.{{0,{window_size}}}"
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    return [match.strip().replace('\n', ' ') for match in matches]

sweden_context = extract_keyword_context(corpus_text, "Sweden", window_size=40)
saab_context = extract_keyword_context(corpus_text, "Saab", window_size=40)

print(f"\nExtracted {len(sweden_context)} contexts mentioning 'Sweden'. Sample:")
for context in sweden_context[:3]:
    print(f" -> ...{context}...")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


Corpus successfully loaded. Total characters: 441537
Initial State -> Tokens: 84359 | Unique Types: 9744
After Stopword Filter -> Tokens: 54028 | Unique Types: 9546
Vocabulary post-Lemmatization: 8796 types
Vocabulary post-Stemming: 6487 types

Named Entity Counts:
- PERSON: 1886 instances (971 unique)
- ORGANIZATION: 1543 instances (631 unique)
- GPE: 1432 instances (366 unique)

Top 5 Most Frequent Trigrams:
- , '' said: 131
- said : ``: 89
- '' said .: 62
- said . ``: 51
- . However ,: 38

Extracted 9 contexts mentioning 'Sweden'. Sample:
 -> ...d HI Europe, placed the UK third behind Sweden and Ireland for business use of ICT....
 -> ...Germany, Spain, Italy, Switzerland and Sweden will be stored permanently in the UK. A...
 -> ...ncluding the national sides of Ireland, Sweden and Norway. Shares in Umbro were up 1.7...
